# Homework 05: Data Storage

This notebook creates an environment-driven CSV/Parquet storage layer, reloads both snapshots, and validates their schema.

In [1]:
from datetime import datetime, timezone
import os
from pathlib import Path

import pandas as pd
from pandas.api.types import is_datetime64_any_dtype, is_float_dtype, is_integer_dtype
from dotenv import load_dotenv

## 1. Environment-driven paths and source DataFrame

In [2]:
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name != 'homework05':
    candidate = PROJECT_DIR / 'homework' / 'homework05'
    if candidate.is_dir():
        PROJECT_DIR = candidate

load_dotenv(PROJECT_DIR / '.env')
raw_dir = PROJECT_DIR / os.getenv('DATA_DIR_RAW', 'data/raw')
processed_dir = PROJECT_DIR / os.getenv('DATA_DIR_PROCESSED', 'data/processed')

source_path = PROJECT_DIR.parent / 'homework04' / 'data' / 'raw' / 'api_yahoo_SPY_20260819-1659.csv'
if not source_path.exists():
    raise FileNotFoundError(f'Homework 04 source data not found: {source_path}')

spy_df = pd.read_csv(source_path, parse_dates=['Date'])
spy_df['Volume'] = pd.to_numeric(spy_df['Volume'], errors='raise').astype('int64')
spy_df.head()

,Date,Adj Close,Close,High,Low,Open,Volume
0,2025-08-19 00:00:00+00:00,632.798401,639.809998,644.109985,638.479980,643.119995,69750700
1,2025-08-20 00:00:00+00:00,631.117004,638.109985,639.659973,632.950012,639.400024,88890300
2,2025-08-21 00:00:00+00:00,628.585083,635.549988,637.969971,633.809998,636.280029,54805800
3,2025-08-22 00:00:00+00:00,638.238220,645.309998,646.500000,637.250000,637.760010,84083200
4,2025-08-25 00:00:00+00:00,635.429199,642.469971,645.289978,642.349976,644.039978,51274300


## 2. Reusable storage utilities

In [3]:
PARQUET_ENGINE_MESSAGE = (
    "Parquet support requires an optional engine. "
    "Install one with `pip install pyarrow` or `pip install fastparquet`."
)

def write_df(df: pd.DataFrame, path: str | Path, *, index: bool = False) -> Path:
    """Write a DataFrame according to the destination's .csv/.parquet suffix."""
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    suffix = destination.suffix.lower()
    if suffix == '.csv':
        df.to_csv(destination, index=index)
    elif suffix == '.parquet':
        try:
            df.to_parquet(destination, index=index)
        except (ImportError, ModuleNotFoundError) as exc:
            raise RuntimeError(PARQUET_ENGINE_MESSAGE) from exc
    else:
        raise ValueError(f"Unsupported file suffix '{suffix}'. Use .csv or .parquet.")
    return destination

def read_df(path: str | Path, **kwargs) -> pd.DataFrame:
    """Read a DataFrame according to the source's .csv/.parquet suffix."""
    source = Path(path)
    if not source.exists():
        raise FileNotFoundError(f'Data file not found: {source}')
    suffix = source.suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(source, **kwargs)
    if suffix == '.parquet':
        try:
            return pd.read_parquet(source, **kwargs)
        except (ImportError, ModuleNotFoundError) as exc:
            raise RuntimeError(PARQUET_ENGINE_MESSAGE) from exc
    raise ValueError(f"Unsupported file suffix '{suffix}'. Use .csv or .parquet.")

## 3. Save CSV and Parquet snapshots

In [4]:
timestamp = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M')
csv_path = raw_dir / f'spy_{timestamp}.csv'
parquet_path = processed_dir / f'spy_{timestamp}.parquet'

write_df(spy_df, csv_path)
write_df(spy_df, parquet_path)
print(f'CSV: {csv_path.relative_to(PROJECT_DIR)}')
print(f'Parquet: {parquet_path.relative_to(PROJECT_DIR)}')

CSV: data/raw/spy_20260824-1456.csv
Parquet: data/processed/spy_20260824-1456.parquet


## 4. Reload and validate

In [5]:
csv_df = read_df(csv_path, parse_dates=['Date'])
parquet_df = read_df(parquet_path)

def validate_storage(original: pd.DataFrame, csv_copy: pd.DataFrame, parquet_copy: pd.DataFrame) -> dict[str, bool]:
    """Return explicit checks for shape, columns, and critical dtype families."""
    critical = {'Date', 'Close', 'Volume'}
    frames = (original, csv_copy, parquet_copy)
    results = {
        'shapes_match': len({frame.shape for frame in frames}) == 1,
        'critical_columns_present': all(critical.issubset(frame.columns) for frame in frames),
        'Date_is_datetime': all(is_datetime64_any_dtype(frame['Date']) for frame in frames),
        'Close_is_float': all(is_float_dtype(frame['Close']) for frame in frames),
        'Volume_is_integer': all(is_integer_dtype(frame['Volume']) for frame in frames),
    }
    results['all_checks_pass'] = all(results.values())
    return results

validation = validate_storage(spy_df, csv_df, parquet_df)
pd.Series(validation, name='passed').to_frame()

,passed
shapes_match,True
critical_columns_present,True
Date_is_datetime,True
Close_is_float,True
Volume_is_integer,True
all_checks_pass,True


In [6]:
assert validation['all_checks_pass'], validation
print('Original:', spy_df.shape, spy_df[['Date', 'Close', 'Volume']].dtypes.to_dict())
print('CSV:     ', csv_df.shape, csv_df[['Date', 'Close', 'Volume']].dtypes.to_dict())
print('Parquet: ', parquet_df.shape, parquet_df[['Date', 'Close', 'Volume']].dtypes.to_dict())
print('All storage validation checks passed.')

Original: (252, 7) {'Date': datetime64[ns, UTC], 'Close': dtype('float64'), 'Volume': dtype('int64')}
CSV:      (252, 7) {'Date': datetime64[ns, UTC], 'Close': dtype('float64'), 'Volume': dtype('int64')}
Parquet:  (252, 7) {'Date': datetime64[ns, UTC], 'Close': dtype('float64'), 'Volume': dtype('int64')}
All storage validation checks passed.


## Storage choice

CSV is kept as the raw interchange snapshot because it is broadly readable and easy to inspect. Parquet is the processed format because it preserves types and is more efficient for analytical workloads. The validation intentionally checks dtype families for critical fields: exact datetime metadata may vary between storage formats while the semantic type remains correct.